In [2]:
import argparse
import itertools
import torch
import triton
import triton.language as tl
import triton.profiler as proton
from triton.tools.tensor_descriptor import TensorDescriptor
from contextlib import contextmanager
from typing import Optional

hook 与 autotune 

hook 可以作用 autotune的每一次测试 


In [84]:
# 标准输出清零pre_hook，适配你的矩阵乘C输出
# def pre_hook(nargs):
#     # nargs是kernel所有入参字典，拿到输出张量C

#     output = nargs["C_ptr"]
    
#     a = nargs["A_ptr"]
#     b = nargs["B_ptr"]
#     # 例如在此处进行精度检测
#     print(f"{(torch.matmul(a,b) - output).abs().max() < 1e-2} from hook")
    
#     output.zero_()  # GPU原地清零，每轮测试前必清

@triton.autotune(
    configs=[triton.Config(
            {
                'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 1
            },num_stages=s, num_warps=w,pre_hook=None)
            # }, num_stages=s, num_warps=w, pre_hook=pre_hook)
        for BM in [128]  #
        for BN in [128, 64]  #
        for BK in [64, 128] #
        for s in [1,2,4,8] 
        for w in [2,4,8]  ],
    key=["M", "N", "K"], # 作为调优的标识符 
)
@triton.jit 
def MyGEMM_Kernel_v1(
    A_ptr,B_ptr,C_ptr,
    M,N,K,
    stride_Am,stride_Ak,
    stride_Bk,stride_Bn,
    stride_Cm,stride_Cn,
    Alpha,Beta,
    BLOCK_SIZE_M:tl.constexpr,
    BLOCK_SIZE_N:tl.constexpr,
    BLOCK_SIZE_K:tl.constexpr,
    GROUP_SIZE_M:tl.constexpr):


    
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M,BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N,BLOCK_SIZE_N)
    num_block_per_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_block_per_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(GROUP_SIZE_M,num_pid_m-first_pid_m)
    pid_m = (pid % num_block_per_group) % group_size_m + first_pid_m
    pid_n = (pid % num_block_per_group) // group_size_m
    #此处可以封装 

    
    '''
    Grouped ordering  
    取代了 ：      
            pid_m = tl.program_id(0)
            pid_n = tl.program_id(1)
    '''
    
    offsets_m = pid_m * BLOCK_SIZE_M + tl.arange(0,BLOCK_SIZE_M)
    offsets_n = pid_n * BLOCK_SIZE_N + tl.arange(0,BLOCK_SIZE_N)
    mask_m = offsets_m < M
    mask_n = offsets_n < N
    offsets_k = tl.arange(0,BLOCK_SIZE_K)
    A_ptrs = A_ptr + offsets_m[:,None] * stride_Am + offsets_k[None,:] * stride_Ak
    B_ptrs = B_ptr + offsets_k[:,None] * stride_Bk + offsets_n[None,:] * stride_Bn
    acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)

    for k in range(0,K,BLOCK_SIZE_K):
        mask_k = offsets_k < K - k
        A = tl.load(A_ptrs,mask = mask_m[:,None] & mask_k[None,:] ,other = 0.0) 
        B = tl.load(B_ptrs,mask = mask_k[:,None] & mask_n[None,:] ,other = 0.0)
        acc = tl.dot(A,B,acc)
        A_ptrs += BLOCK_SIZE_K * stride_Ak
        B_ptrs += BLOCK_SIZE_K * stride_Bk
        
    C_ptrs = C_ptr + offsets_m[:,None] * stride_Cm + offsets_n[None,:] * stride_Cn
    c_old = tl.load(C_ptrs, mask=mask_m[:,None] & mask_n[None,:], other=0.0)
    acc = acc * Alpha + Beta * c_old
    
    tl.store(C_ptrs,acc.to(tl.float16),mask = mask_m[:,None] & mask_n[None,:])

def matmul_v1(a, b,Alpha,Beta):
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.dtype == b.dtype, "Incompatible dtypes"
    M, K = a.shape
    K, N = b.shape
    dtype = a.dtype

    C = torch.empty((M, N), device=a.device, dtype=dtype)
    grid = lambda META: (triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"]), )

    MyGEMM_Kernel_v1[grid](
        a, b, c,
        M, N, K,
        # ========== 这里必须加 int()！！ ==========
        int(a.stride(0)), int(a.stride(1)),
        int(b.stride(0)), int(b.stride(1)),
        int(c.stride(0)), int(c.stride(1)),
        Alpha,Beta,
    )
    return c

In [85]:
a = torch.tensor([[1,2,3],[4,5,6]],device = 'cuda',dtype=torch.float16)
b = torch.tensor([[3,4],[5,6],[7,8]],device = 'cuda',dtype=torch.float16)
Alpha = 1.0
Beta = 0.0
c = torch.matmul(a,b,)
c_triton = matmul_v1(a,b,Alpha,Beta)
print(f"{(c- c_triton).abs().max() < 1e-2}")

True


_第一次执行会输出很多，但第二次执行就不需要再测试了_

### 使用 tma
####  创建Tensordescriptor
    dummy_block = [1, 1]
    a_desc = TensorDescriptor.from_tensor(a, dummy_block)
    b_desc = TensorDescriptor.from_tensor(b, dummy_block)
    c_desc = TensorDescriptor.from_tensor(c, dummy_block)
#### 注意形状!! （使用pre_hook）
    nargs["a_desc"].block_shape = [BLOCK_M, BLOCK_K]
    nargs["b_desc"].block_shape = [BLOCK_N, BLOCK_K]
#### 读取与存储数据 
        a = a_desc.load([offsets_m,offsets_k])
        b = b_desc.load([offsets_n,offsets_k])
        c_desc.store([offsets_m, offsets_n], c)
#### 注意B需要传入转置进行计算 （相当于直接拿张量计算）
        acc = tl.dot(a,b.T,acc)
### dummy_block 参数的作用

dummy_block 在这里作为 block_shape 传递给 TensorDescriptor.from_tensor。block_shape 是描述每个块在内存中的尺寸，它通常在 Triton 中用于 分块计算，帮助对大矩阵进行分块处理以提高效率。block_shape 确定了每次加载到寄存器中的数据的块大小。

### WARP_SPECIALIZE 作用
在 Triton 中，WARP_SPECIALIZE 是一个参数，用于控制 warp-level specialization（即线程块的优化）是否启用。它是一个特定于 NVIDIA GPU（特别是 CUDA）的特性，旨在 优化矩阵乘法等计算密集型操作，通过更细粒度地控制线程块和 warp 的行为来提高性能。

### num_stages：控制计算的流水阶段数
num_stages 参数用于指定 kernel 计算的流水化阶段数。它控制了计算的并行程度，尤其是如何将计算过程拆分成多个阶段并行执行。这个参数对于提高 GPU 的计算效率和资源利用率非常重要。

### num_warps：控制每个阶段的 warp 数量
num_warps 参数控制每个计算阶段中的 warp 数量。warp 是 GPU 上执行的基本线程单位，由 32 个线程组成。在 Triton 中，num_warps 参数决定了每个阶段会分配多少个 warp 来执行计算。

In [81]:
from triton.tools.tensor_descriptor import TensorDescriptor
def pre_hook(nargs):
    EPILOGUE_SUBTILE = nargs.get("EPILOGUE_SUBTILE", False)
    BLOCK_M = nargs["BLOCK_SIZE_M"]
    BLOCK_N = nargs["BLOCK_SIZE_N"]
    BLOCK_K = nargs["BLOCK_SIZE_K"]
    nargs["a_desc"].block_shape = [BLOCK_M, BLOCK_K]
    nargs["b_desc"].block_shape = [BLOCK_N, BLOCK_K]
    if EPILOGUE_SUBTILE:
        nargs["c_desc"].block_shape = [BLOCK_M, BLOCK_N // 2]
    else:
        nargs["c_desc"].block_shape = [BLOCK_M, BLOCK_N]
        
@triton.autotune(
    configs=[triton.Config(
            {
                'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 1
            },num_stages=s, num_warps=w,pre_hook=pre_hook)
            # }, num_stages=s, num_warps=w, pre_hook=pre_hook)
        for BM in [128]  #
        for BN in [128, 64]  #
        for BK in [64, 128] #
        for s in [1,2,4,8] 
        for w in [2,4,8]  ],
    key=["M", "N", "K", "WARP_SPECIALIZE"],
)
@triton.jit 
def MyGEMM_Kernel_v2_tma(
    a_desc, b_desc, c_desc, 
    M,N,K,
    Alpha,Beta,
    BLOCK_SIZE_M:tl.constexpr,
    BLOCK_SIZE_N:tl.constexpr,
    BLOCK_SIZE_K:tl.constexpr,
    GROUP_SIZE_M:tl.constexpr,
    FP8_OUTPUT: tl.constexpr,  #
    WARP_SPECIALIZE: tl.constexpr ):
    # 计算时采用 高精度，存储时采用低精度
    dtype = tl.float8e4nv if FP8_OUTPUT else tl.float16

    pid = tl.program_id(axis = 0)
    num_pid_m = tl.cdiv(M,BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N,BLOCK_SIZE_N)
    num_block_per_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_block_per_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(GROUP_SIZE_M,num_pid_m-first_pid_m)
    pid_m = (pid % num_block_per_group) % group_size_m + first_pid_m
    pid_n = (pid % num_block_per_group) // group_size_m

    offsets_m = pid_m * BLOCK_SIZE_M 
    offsets_n = pid_n * BLOCK_SIZE_N 
    k_tiles = tl.cdiv(K,BLOCK_SIZE_K)
    acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)

    for k in tl.range(k_tiles,warp_specialize= WARP_SPECIALIZE):
        offsets_k = k * BLOCK_SIZE_K
        a = a_desc.load([offsets_m,offsets_k])
        b = b_desc.load([offsets_n,offsets_k])
        acc = tl.dot(a, b.T, acc)
        
    c_old = c_desc.load([offsets_m,offsets_n]).to(tl.float32)
    acc = acc * Alpha + Beta * c_old
    c = acc.to(dtype)
    c_desc.store([offsets_m, offsets_n], c)

def matmul_v2_tma(a, b, Alpha,Beta,warp_specialize: bool):
    # Check constraints.
    assert a.shape[1] == b.shape[1], "Incompatible dimensions"  # b is transposed
    assert a.dtype == b.dtype, "Incompatible dtypes"

    M, K = a.shape
    N, K = b.shape
    dtype = a.dtype

    c = torch.empty((M, N), device=a.device, dtype=dtype)

    # A dummy block value that will be overwritten when we have the real block size
    dummy_block = [1, 1]
    a_desc = TensorDescriptor.from_tensor(a, dummy_block)
    b_desc = TensorDescriptor.from_tensor(b, dummy_block)
    c_desc = TensorDescriptor.from_tensor(c, dummy_block)
    
    def grid(META):
        BLOCK_M = META["BLOCK_SIZE_M"]
        BLOCK_N = META["BLOCK_SIZE_N"]
        return (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N), )

    MyGEMM_Kernel_v2_tma[grid](
        a_desc, b_desc, c_desc,  #
        M, N, K, Alpha,Beta,
        FP8_OUTPUT=dtype == torch.float8_e4m3fn,  #
        WARP_SPECIALIZE=warp_specialize,  #
    )
    return c

## TMA 的 TensorDescriptor 要求 stride 满足 16-byte 对齐


In [82]:
a = torch.randn((64, 16), device='cuda', dtype=torch.float16)
b = torch.randn((16, 48), device='cuda', dtype=torch.float16)
Alpha = 1.0
Beta = 0.0
# wrapper 约定传入的是转置后的 b，即 shape (N, K)
c_triton_tma = matmul_v2_tma(a.contiguous(), b.T.contiguous(),Alpha,Beta,True)
c_torch = torch.matmul(a, b)
print(torch.allclose(c_triton_tma, c_torch, atol=1e-2, rtol=1e-2))

True


### Persistent GEMM：

只启动 NUM_SMS 个 Thread Block（刚好占满 GPU）

这些线程块  **一直活着，不退出、不调度、不释放**

内核内部自己 **分发不同的任务（tile_id）** 给它们循环计算

### 使用注意事项 ：

#### 计算GPU可用 线程块的个数 
    NUM_SMS = torch.cuda.get_device_properties("cuda").multi_processor_count
#### grid 大小计算 
    grid = lambda META: (min(NUM_SMS, triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"])), )
    

In [93]:
@triton.jit
def _compute_pid(tile_id, num_pid_in_group, num_pid_m, GROUP_SIZE_M, NUM_SMS):
    # 根据tile计算出pid_m 与 pid_n的坐标 
    group_id = tile_id // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + (tile_id % group_size_m)
    pid_n = (tile_id % num_pid_in_group) // group_size_m
    return pid_m, pid_n
    
@triton.autotune(
    configs=[triton.Config(
                {
                    'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": GM
                },num_stages=s, num_warps=w,pre_hook=None)
            for BM in [128]  #
            for BN in [128, 64]  #
            for BK in [64, 128] #
            for s in [1,2,4,8] 
            for w in [2,4,8]
            for GM in [1,2,4,8]
            ],
        key=["M", "N", "K"],
)
@triton.jit
def MyGEMM_Kernel_v3_persistent(a_ptr, b_ptr, c_ptr,  #
                             M, N, K,  #
                             stride_am, stride_ak,  #
                             stride_bk, stride_bn,  #
                             stride_cm, stride_cn,  #
                             Alpha,Beta,
                             BLOCK_SIZE_M: tl.constexpr,  #
                             BLOCK_SIZE_N: tl.constexpr,  #
                             BLOCK_SIZE_K: tl.constexpr,  #
                             GROUP_SIZE_M: tl.constexpr,  #
                             NUM_SMS: tl.constexpr,  #
                             ):
    start_pid = tl.program_id(axis=0)
    
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    k_tiles = tl.cdiv(K, BLOCK_SIZE_K)
    num_tiles = num_pid_m * num_pid_n
    # 每个 Thread Block 负责 输出结果的 的一块 ，但是GPU没有足够的线程块，所以会产生 线程块的调度 
    # 当引入 presistent GEMM时 共有 NUM_SMS 个Tread Block 此时，线程块不再去调度，而是分发不同的任务

    # NOTE: There is currently a bug in blackwell pipelining that means it can't handle a value being
    # used in both the prologue and epilogue, so we duplicate the counters as a work-around.
    tile_id_c = start_pid - NUM_SMS

    offs_k_for_mask = tl.arange(0, BLOCK_SIZE_K)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n

    for tile_id in tl.range(start_pid, num_tiles, NUM_SMS, flatten=True):
        # 每个显存块Block负责 矩阵上 第i i+NUM_SMS i+2*NUM_SMS....的数据
        
        pid_m, pid_n = _compute_pid(tile_id, num_pid_in_group, num_pid_m, GROUP_SIZE_M, NUM_SMS)
        
        start_m = pid_m * BLOCK_SIZE_M
        start_n = pid_n * BLOCK_SIZE_N
        offs_am = start_m + tl.arange(0, BLOCK_SIZE_M)
        offs_bn = start_n + tl.arange(0, BLOCK_SIZE_N)
        offs_am = tl.where(offs_am < M, offs_am, 0)
        offs_bn = tl.where(offs_bn < N, offs_bn, 0)
        offs_am = tl.max_contiguous(tl.multiple_of(offs_am, BLOCK_SIZE_M), BLOCK_SIZE_M)
        offs_bn = tl.max_contiguous(tl.multiple_of(offs_bn, BLOCK_SIZE_N), BLOCK_SIZE_N)
        
        accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
        
        for ki in range(k_tiles):
            offs_k = ki * BLOCK_SIZE_K + tl.arange(0, BLOCK_SIZE_K)
            #下一步应该换成a_desc,b_desc (本设备不支持TMA)
            a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
            b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

            a = tl.load(a_ptrs, mask=offs_k_for_mask[None, :] < K - ki * BLOCK_SIZE_K, other=0.0)
            b = tl.load(b_ptrs, mask=offs_k_for_mask[:, None] < K - ki * BLOCK_SIZE_K, other=0.0)
            accumulator = tl.dot(a, b, accumulator)
            
        tile_id_c += NUM_SMS
        pid_m, pid_n = _compute_pid(tile_id_c, num_pid_in_group, num_pid_m, GROUP_SIZE_M, NUM_SMS)
        offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
        offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
        c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
        c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
        
        c_old = tl.load(c_ptrs,mask = c_mask,other = 0.0)
        accumulator = Alpha * accumulator + c_old * Beta
        
        if (c_ptr.dtype.element_ty == tl.float8e4nv):
            c = accumulator.to(tl.float8e4nv)
        else:
            c = accumulator.to(tl.float16)
        tl.store(c_ptrs, c, mask=c_mask)


def matmul_v3_persistent(a, b,Alpha,Beta):
    # Check constraints.
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.dtype == b.dtype, "Incompatible dtypes"
    NUM_SMS = torch.cuda.get_device_properties("cuda").multi_processor_count
    M, K = a.shape
    K, N = b.shape
    dtype = a.dtype
    # Allocates output.
    c = torch.empty((M, N), device=a.device, dtype=dtype)
    # 1D launch kernel where each block gets its own program.
    grid = lambda META: (min(NUM_SMS, triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"])), )
    MyGEMM_Kernel_v3_persistent[grid](
        a, b, c,  #
        M, N, K,  #
        a.stride(0), a.stride(1),  #
        b.stride(0), b.stride(1),  #
        c.stride(0), c.stride(1),  #
        Alpha,Beta,
        NUM_SMS=NUM_SMS,  #
    )
    return c

In [94]:
a = torch.randn((16, 8), device='cuda', dtype=torch.float16)
b = torch.randn((8, 8), device='cuda', dtype=torch.float16)

Alpha = 1.0
Beta = 0.0
# wrapper 约定传入的是转置后的 b，即 shape (N, K)
c_triton_tma = matmul_v3_persistent(a, b,Alpha,Beta)

c_torch = torch.matmul(a, b)
print(torch.allclose(c_triton_tma, c_torch, atol=1e-2, rtol=1e-2))

True


In [96]:
def pre_hook(nargs):
    EPILOGUE_SUBTILE = nargs.get("EPILOGUE_SUBTILE", False)
    BLOCK_M = nargs["BLOCK_SIZE_M"]
    BLOCK_N = nargs["BLOCK_SIZE_N"]
    BLOCK_K = nargs["BLOCK_SIZE_K"]
    nargs["a_desc"].block_shape = [BLOCK_M, BLOCK_K]
    nargs["b_desc"].block_shape = [BLOCK_N, BLOCK_K]
    if EPILOGUE_SUBTILE:
        nargs["c_desc"].block_shape = [BLOCK_M, BLOCK_N // 2]
    else:
        nargs["c_desc"].block_shape = [BLOCK_M, BLOCK_N]
        
def matmul_tma_persistent_get_configs(pre_hook=None):
    return [
        triton.Config(
            {
                'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 1
            }, num_stages=s, num_warps=w, pre_hook=pre_hook)  #
        for BM in [128]  #
        for BN in [128, 64]  #
        for BK in [64, 32]  #
        for s in ([1,2])  #
        for w in [2,4]  #
    ]

@triton.jit 
def _compute_PID(tile_id,num_pid_in_group,num_pid_m,GROUP_SIZE_M):
    # 根据数据所在 tile计算出tile_m 与 tile_n的坐标 
    group_id = tile_id // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((tile_id % num_pid_in_group) % group_size_m)
    pid_n = (tile_id % num_pid_in_group) // group_size_m
    return pid_m, pid_n
@triton.autotune(
    configs=matmul_tma_persistent_get_configs(pre_hook=matmul_tma_set_block_size_hook),
    key=["M", "N", "K", "WARP_SPECIALIZE"],
)
@triton.jit
def MyGEMM_Kernel_v4_persistent_tma(
    a_desc, b_desc, c_desc, 
    M,N,K,
    Alpha,Beta,
    BLOCK_SIZE_M:tl.constexpr,
    BLOCK_SIZE_N:tl.constexpr,
    BLOCK_SIZE_K:tl.constexpr,
    GROUP_SIZE_M:tl.constexpr,
    FP8_OUTPUT: tl.constexpr,  #
    NUM_SMS: tl.constexpr,  #
    WARP_SPECIALIZE: tl.constexpr,
    FLATTEN:tl.constexpr
):
    dtype = tl.float8e4nv if FP8_OUTPUT else tl.float16 # 存储类型 
    # 计算类型 ：float32
    pid = tl.program_id(axis = 0) # pid 总共就 NUM_SMS 个 
    num_pid_m = tl.cdiv(M,BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N,BLOCK_SIZE_N)
    # k_tiles = tl.cdiv(K, BLOCK_SIZE_K) = range(0,K,BLOCK_SIZE_K)
    num_tiles = num_pid_m * num_pid_n
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    #  NUM_SMS个 线程块 负责 num_tiles的数据量输出
    tile_id_c = pid - NUM_SMS
    
    for tile_id in tl.range(pid,num_tiles,NUM_SMS,flatten=True, warp_specialize=WARP_SPECIALIZE):
        # 取 tile_m,tile_n 0 - 4 - 8 - 12 - 16 
        tile_m,tile_n = _compute_PID(tile_id,num_pid_in_group,num_pid_m,GROUP_SIZE_M)
        offset_m = tile_m * BLOCK_SIZE_M
        offset_n = tile_n * BLOCK_SIZE_N 
        
        acc = tl.zeros((BLOCK_SIZE_M,BLOCK_SIZE_N),dtype = tl.float32)
        for rk in tl.range(0,K,BLOCK_SIZE_K,warp_specialize = WARP_SPECIALIZE):
            a = a_desc.load([offset_m,rk])
            b = b_desc.load([offset_n,rk])
            acc = tl.dot(a,b.T,acc)
            
        tile_id_c += NUM_SMS
        # 存tile_m,tile_n 0 - 4 - 8 - 12 - 16
        tile_m,tile_n = _compute_PID(tile_id_c,num_pid_in_group,num_pid_m,GROUP_SIZE_M)
        offset_m = tile_m * BLOCK_SIZE_M
        offset_n = tile_n * BLOCK_SIZE_N 
        c_old = c_desc.load([offset_m,offset_n]).to(tl.float32)
        acc = acc*Alpha + Beta*c_old
        c_desc.store([offset_m, offset_n], acc.to(dtype))
        
def matmul_v4_tma_persistent(a, b,Alpha,Beta,warp_specialize: bool):
    # Check constraints.
    assert a.shape[1] == b.shape[1], "Incompatible dimensions"  # b is transposed
    assert a.dtype == b.dtype, "Incompatible dtypes"

    M, K = a.shape
    N, K = b.shape
    dtype = a.dtype

    c = torch.empty((M, N), device=a.device, dtype=dtype)
    
    NUM_SMS = torch.cuda.get_device_properties("cuda").multi_processor_count
    # A dummy block value that will be overwritten when we have the real block size
    def alloc_fn(size: int, alignment: int, stream: Optional[int]):
        return torch.empty(size, device="cuda", dtype=torch.int8)

    triton.set_allocator(alloc_fn)
    dummy_block = [1, 1]
    a_desc = TensorDescriptor.from_tensor(a, dummy_block)
    b_desc = TensorDescriptor.from_tensor(b, dummy_block)
    c_desc = TensorDescriptor.from_tensor(c, dummy_block)
    
    # Hopper warpspec doesn't work with flatten
    flatten =  True
    grid = lambda META: (min(NUM_SMS, triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"])), )
    MyGEMM_Kernel_v4_persistent_tma[grid](
        a_desc,
        b_desc,
        c_desc,
        M,
        N,
        K,
        Alpha,
        Beta,
        FP8_OUTPUT=False,
        NUM_SMS=NUM_SMS,
        WARP_SPECIALIZE=warp_specialize,
        FLATTEN=flatten,
    )
    return c

    

In [99]:
a = torch.randn((16, 8), device='cuda', dtype=torch.float16)
b = torch.randn((8, 8), device='cuda', dtype=torch.float16)
Alpha = 1.0
Beta = 0.0
# wrapper 约定传入的是转置后的 b，即 shape (N, K)
c_triton_tma = matmul_v4_tma_persistent(a.contiguous(), b.T.contiguous(),Alpha,Beta, True)
c_torch = torch.matmul(a, b)
print(torch.allclose(c_triton_tma, c_torch, atol=1e-2, rtol=1e-2))

True


### 便捷操作：封装参数

In [100]:
def pre_hook(nargs):
    EPILOGUE_SUBTILE = nargs.get("EPILOGUE_SUBTILE", False)
    BLOCK_M = nargs["BLOCK_SIZE_M"]
    BLOCK_N = nargs["BLOCK_SIZE_N"]
    BLOCK_K = nargs["BLOCK_SIZE_K"]
    nargs["a_desc"].block_shape = [BLOCK_M, BLOCK_K]
    nargs["b_desc"].block_shape = [BLOCK_N, BLOCK_K]
    if EPILOGUE_SUBTILE:
        nargs["c_desc"].block_shape = [BLOCK_M, BLOCK_N // 2]
    else:
        nargs["c_desc"].block_shape = [BLOCK_M, BLOCK_N]
def matmul_tma_persistent_get_configs(pre_hook=None):
    return [
        triton.Config(
            {
                'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 1
            }, num_stages=s, num_warps=w, pre_hook=pre_hook)  #
        for BM in [128]  #
        for BN in [128, 64]  #
        for BK in [64, 32]  #
        for s in ([1,2])  #
        for w in [2,4]  #
    ]

In [ ]:
import torch
import triton
import matplotlib.pyplot as plt
# ==========================================
# 假设这里已经包含了你修改好的 matmul 和 matmul_tma 函数
# def matmul(a, b): ...
# def matmul_tma(a, b, warp_specialize): ...
# ==========================================

def run_benchmark():
    # 定义测试的矩阵维度，从小型矩阵到中大型矩阵
    # 保证 M = N = K 以便简化计算
    sizes = [512, 1024, 2048, 4096, 8192]
    
    # 初始化记录列表
    results = {
        "PyTorch (cuBLAS)": {"tflops": [], "bw": []},
        "Triton V1": {"tflops": [], "bw": []},
        "Triton V2 (TMA)": {"tflops": [], "bw": []},
        "Triton V3 (persistent)": {"tflops": [], "bw": []},
        "Triton V4 (TMA + persistent)": {"tflops": [], "bw": []},
    }

    print("开始运行 Benchmark (可能需要几分钟进行 Autotune 编译)...")
    
    for M in sizes:
        N = K = M
        print(f"Testing size: {M}x{K}x{N}...")
        
        # 强制使用 float16 激活 Tensor Cores
        a = torch.randn((M, K), device='cuda', dtype=torch.float16)
        b = torch.randn((K, N), device='cuda', dtype=torch.float16)
        
        # TMA 版本的 b 需要转置 (因为你代码里的 assert 逻辑 b.shape[1] == a.shape[1])
        b_tma = b.t().contiguous() 
        
        # 计算理论浮点运算次数 (FLOPs) 和 理论显存读写量 (Bytes)
        flops = 2 * M * N * K
        # 读A, 读B, 写C，fp16 占 2 Bytes
        bytes_accessed = 2 * (M * K + K * N + M * N)
        Alpha = 1.0
        Beta = 0.0
        # 定义测试闭包
        func_torch = lambda: torch.matmul(a, b)
        func_v1    = lambda: matmul_v1(a, b,Alpha,Beta)
        func_v2    = lambda: matmul_v2_tma(a.contiguous(), b.T.contiguous(),Alpha,Beta,True)
        func_v3    = lambda: matmul_v3_persistent(a, b,Alpha,Beta)
        func_v4    = lambda: matmul_v4_tma_persistent(a.contiguous(), b.T.contiguous(),Alpha,Beta, True)
        
        # 使用 Triton 内置的精准测速工具 (自动处理 Warmup 和同步)
        ms_torch = triton.testing.do_bench(func_torch)
        ms_v1    = triton.testing.do_bench(func_v1)
        ms_v2    = triton.testing.do_bench(func_v2)
        ms_v3    = triton.testing.do_bench(func_v3)
        ms_v4    = triton.testing.do_bench(func_v4)
        # 将耗时转化为 TFLOPS 和 GB/s
        for name, ms in zip(results.keys(), [ms_torch, ms_v1, ms_v2,ms_v3,ms_v4]):
            sec = ms / 1000.0
            results[name]["tflops"].append((flops / sec) / 1e12)
            results[name]["bw"].append((bytes_accessed / sec) / 1e9)

    # ================= 绘图部分 =================
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    x_labels = [str(s) for s in sizes]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c','#4bda11','#8b1a71']
    markers = ['o', 's', '^','*','.']

    # 1. 绘制 TFLOPS (算力对比)
    for idx, (name, data) in enumerate(results.items()):
        ax1.plot(x_labels, data["tflops"], label=name, color=colors[idx], 
                 marker=markers[idx], linewidth=2, markersize=8)
    
    ax1.set_title("Performance Comparison (TFLOPS)\nHigher is better", fontsize=14)
    ax1.set_xlabel("Matrix Size (M=N=K)", fontsize=12)
    ax1.set_ylabel("TFLOPS", fontsize=12)
    ax1.legend(fontsize=11)
    
    # 2. 绘制 Bandwidth (有效带宽对比)
    for idx, (name, data) in enumerate(results.items()):
        ax2.plot(x_labels, data["bw"], label=name, color=colors[idx], 
                 marker=markers[idx], linewidth=2, markersize=8)
                 
    ax2.set_title("Effective Bandwidth (GB/s)\nHigher is better", fontsize=14)
    ax2.set_xlabel("Matrix Size (M=N=K)", fontsize=12)
    ax2.set_ylabel("Bandwidth (GB/s)", fontsize=12)
    ax2.legend(fontsize=11)

    plt.tight_layout()
    plt.savefig("gemm_benchmark_comparison.png", dpi=300)
    print("测试完成！图表已保存为 'gemm_benchmark_comparison.png'")
    plt.show()

if __name__ == "__main__":
    run_benchmark()

开始运行 Benchmark (可能需要几分钟进行 Autotune 编译)...
Testing size: 512x512x512...
